# Bank Fraud Detection — Group Assignment 2, Milestone 5 (Final)
## Model Evaluation, Visualization & Decision Storytelling

**Group 8:** Isaac Irabor · Jaswanth Kumar Reddy Chittem · Md Faiaz · Yokishan Krishnakumar
**Course:** Data Science for Artificial Intelligence • MAI103 **Submitted to:** Prof. Elnaz Delpisheh
**Dataset:** `banking_transactions.csv` — Kaggle, *Banking Fraud Detection and Risk Analytics Dataset*

This notebook is the final stage of our project. Milestone 4 compared logistic regression with and without class-imbalance handling. Here we freeze a final model, evaluate it honestly on held-out data, explain what it learned, and turn the results into recommendations a bank could act on.

**How to run (Google Colab):** `Runtime → Run all`. The data loads from a local/Drive path, from Kaggle through `kagglehub`, or through a file-upload prompt. Every number quoted in the interpretation boxes is **computed at runtime** — nothing is typed in by hand — so the story always matches the outputs. All figures are saved to `ms5_figures/` and the summary tables to `ms5_outputs/`, ready for the PDF report and the slides.

### Where each assignment requirement is answered

| Assignment requirement | Section |
|---|---|
| Part 1 · Q1 Problem and why it matters | §2 |
| Part 1 · Q2 Key EDA findings (only those that drive decisions) | §3 |
| Part 1 · Q3 Final model selection and comparison | §5 |
| Part 1 · Q4 Model performance | §6 – §7 |
| Part 1 · Q5 Insights: what the model learned, feature effects | §8 |
| Part 1 · Q6 Recommendations | §10 |
| Part 1 · Q7 Limitations and risks | §9, §11 |
| Part 2 · Evaluation visualizations (confusion matrix, ROC, PR, error analysis) | §6, §7, §9 |
| Part 2 · Explainability visualizations (SHAP, coefficients, permutation importance) | §8 |
| Part 3 · Real-world use, monitoring, risks, ethics, limitations, future work | §10 – §11 |
| Deliverables · report & slide material | §12 (auto-generated summary + exported figures) |

### What changed since Milestone 4

We reviewed our Milestone 4 notebook before building on it and corrected several issues so that the final numbers can be trusted:

1. **Scaling mismatch fixed.** One validation cell scored models that were trained on *scaled* data using *unscaled* validation features. In Milestone 5, scaling lives **inside every pipeline**, so raw data goes in everywhere and this mistake cannot happen.
2. **Scaler overwrite removed.** The shared `StandardScaler` was re-fitted on train + validation after the train-only fit. That cell is gone; each pipeline owns its own scaler, fitted on training rows only.
3. **Hard-coded results replaced.** The "before vs after SMOTE" table was typed into markdown, and the tuned-model description mixed the L1 grid-search model with the validation-tuned L2 model. Every number is now computed in code.
4. **Test set protected.** A grid-search model was scored on the test set *before* the validation-based selection step. Now the test set is used only after the model **and** the decision threshold are frozen.
5. **Clean-up.** Duplicate cells (models, tuning and sensitivity checks fitted twice) were removed, and the confusion-matrix title no longer says "Balanced Class Weights" regardless of which model was chosen.

**New in Milestone 5:** a categorical-feature experiment (our Milestone 4 "next step"), a non-linear benchmark, validation-tuned decision thresholds, cost and workload analysis, calibration, SHAP + coefficient + permutation explainability, error and segment (fairness) analysis, repeated cross-validation, a drift-monitoring demonstration, and an auto-generated summary for the report and slides.

## 1. Setup

In [ ]:
# Install only what is missing (Colab usually already has both packages)
import importlib.util, subprocess, sys
for pip_name, module_name in [("imbalanced-learn", "imblearn"), ("shap", "shap")]:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

In [ ]:
import os, glob, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline as sk_make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, brier_score_loss)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

warnings.filterwarnings("ignore", message=".*IProgress not found.*")
try:
    import shap
    HAS_SHAP = True
except Exception as exc:
    HAS_SHAP = False
    print("SHAP could not be imported; permutation importance and coefficients will still run.", exc)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*penalty.*deprecated.*")   # newer scikit-learn renames `penalty`
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
FIG_DIR, OUT_DIR = "ms5_figures", "ms5_outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
FIGURES = []

def save_fig(name, fig=None):
    """Save the current figure for the report/slides."""
    fig = fig or plt.gcf()
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=160, bbox_inches="tight")
    if path not in FIGURES:
        FIGURES.append(path)

def md(text):
    display(Markdown(text))

def pct(x):
    return f"{100 * x:.1f}%"

## 2. The problem — why this matters (Part 1, Q1)

Card and account fraud costs banks and customers money, time and trust. A bank cannot review every transaction by hand, so it needs a way to **rank transactions by fraud risk** and send only the riskiest ones to analysts.

Two different mistakes are possible, and they do not cost the same:

- **Missed fraud (false negative):** the money is lost, the customer is harmed, and the fraud pattern keeps working.
- **False alert (false positive):** a legitimate customer may be delayed or declined, and an analyst spends time on a case that was fine.

Our task is therefore not "maximize accuracy". It is: **catch as much fraud as possible while keeping the number of alerts small enough for a review team to handle.** We measure success with recall (share of fraud caught), precision (share of alerts that are real fraud), F1, and PR-AUC (ranking quality that focuses on the fraud class). The model is designed to **support** analysts, not to block customers automatically.

## 3. Data and the EDA findings that shaped our decisions (Part 1, Q2)

We only keep the EDA findings that changed a modelling decision. The full EDA is in earlier milestones.

In [ ]:
DATA_FILE = "banking_transactions.csv"
KAGGLE_DATASET = "deepeshkansotia/banking-fraud-detection-and-risk-analytics-dataset"

def load_data():
    """Load the CSV from (1) a local/Drive path, (2) Kaggle via kagglehub, or (3) a Colab upload."""
    for path in [DATA_FILE, f"/content/{DATA_FILE}", f"/content/drive/MyDrive/{DATA_FILE}"]:
        if os.path.exists(path):
            return pd.read_csv(path), path
    try:
        import kagglehub
        folder = kagglehub.dataset_download(KAGGLE_DATASET)
        csvs = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
        if csvs:
            return pd.read_csv(csvs[0]), csvs[0]
    except Exception as exc:
        print(f"Kaggle download not available ({type(exc).__name__}); falling back to upload.")
    from google.colab import files
    uploaded = files.upload()  # choose banking_transactions.csv
    name = next(iter(uploaded))
    return pd.read_csv(name), name

df, source = load_data()
print("Loaded from:", source)
print("Data size:", df.shape)
display(df.head())

In [ ]:
TARGET = "fraud_flag"
ID_COLS = ["transaction_id"]          # row identifiers carry no predictive meaning

# Robust target conversion (works whether fraud_flag is bool, 0/1 or "True"/"False")
y_all = (df[TARGET].astype(str).str.strip().str.lower()
         .isin(["true", "1", "yes", "fraud"]).astype(int))

features_df = df.drop(columns=[TARGET] + [c for c in ID_COLS if c in df.columns]).copy()
bool_cols = features_df.select_dtypes(include="bool").columns.tolist()
features_df[bool_cols] = features_df[bool_cols].astype(int)

numeric_cols = features_df.select_dtypes(include="number").columns.tolist()
other_cols = [c for c in features_df.columns if c not in numeric_cols]
categorical_cols = [c for c in other_cols if features_df[c].nunique() <= 50]
dropped_cols = [c for c in other_cols if c not in categorical_cols]

quality = pd.Series({
    "Rows": len(df), "Columns": df.shape[1],
    "Missing cells": int(df.isnull().sum().sum()),
    "Duplicate rows": int(df.duplicated().sum()),
    "Fraud rows": int(y_all.sum()), "Non-fraud rows": int((1 - y_all).sum()),
    "Fraud share": f"{y_all.mean():.2%}",
    "Numeric features": len(numeric_cols), "Categorical features": len(categorical_cols),
}, dtype=object)
display(quality.to_frame("Value"))
print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)
if dropped_cols:
    print("High-cardinality text columns not used:", dropped_cols)

### Leakage-safe split (same rows as Milestone 4)

We keep the Milestone 4 design: a stratified **60 / 20 / 20** split into training, validation and test, with the same random seed, so the rows match the previous milestone. Training rows fit the models, validation rows choose the model and the decision threshold, and the test rows are opened **once** at the end. EDA statistics below are computed on **training rows only**, so nothing about the test set influences our decisions.

In [ ]:
X = features_df[numeric_cols + categorical_cols]
y = y_all

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_val)

split_summary = pd.DataFrame({
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Fraud cases": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "Fraud share": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["Training (60%)", "Validation (20%)", "Test (20%)"])
display(split_summary.round(3))

### Figure 1 — Key EDA findings

**What it shows.** (A) How many transactions are fraud. (B) How well each numeric feature separates fraud from non-fraud **on its own**, measured by single-feature ROC-AUC on training rows (0.5 = no signal, 1.0 = perfect); red bars mean higher values go with fraud, blue bars mean lower values do. (C) The fraud rate inside each group of the categorical field whose groups differ most.

**Why it matters.** (A) justifies ignoring accuracy and handling imbalance. (B) tells us whether a simple linear model has enough signal to work with, and acts as a **leakage check**: a single feature that almost perfectly predicts fraud would suggest it was created after the fraud was known. (C) tests our Milestone 4 decision to drop the text fields.

In [ ]:
# (B) Single-feature signal strength, training rows only
uni_rows = []
for c in numeric_cols:
    vals = X_train[c].fillna(X_train[c].median())
    if vals.nunique() < 2:
        continue
    auc = roc_auc_score(y_train, vals)
    uni_rows.append({"feature": c, "AUC": auc, "strength": max(auc, 1 - auc),
                     "direction": "higher → more fraud" if auc >= 0.5 else "lower → more fraud"})
uni_df = pd.DataFrame(uni_rows).sort_values("strength", ascending=False).reset_index(drop=True)

# (C) Fraud rate by category, training rows only
cat_rates = {c: (y_train.groupby(X_train[c]).agg(["mean", "size"])
                 .rename(columns={"mean": "fraud_rate", "size": "n"})
                 .sort_values("fraud_rate", ascending=False))
             for c in categorical_cols}
best_cat = (max(cat_rates, key=lambda c: cat_rates[c]["fraud_rate"].max() - cat_rates[c]["fraud_rate"].min())
            if cat_rates else None)

n_panels = 3 if best_cat else 2
fig, axes = plt.subplots(1, n_panels, figsize=(6.2 * n_panels, 4.8))

counts = y_all.map({0: "Not fraud", 1: "Fraud"}).value_counts().reindex(["Not fraud", "Fraud"])
axes[0].bar(counts.index, counts.values, color=["#4C72B0", "#C44E52"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}\n({v / counts.sum():.1%})", ha="center", va="bottom")
axes[0].set_ylim(0, counts.max() * 1.2)
axes[0].set_title("A. Fraud is the minority class")
axes[0].set_ylabel("Transactions")

top_uni = uni_df.head(10).iloc[::-1]
axes[1].barh(top_uni["feature"], top_uni["strength"],
             color=["#C44E52" if a >= 0.5 else "#4C72B0" for a in top_uni["AUC"]])
axes[1].axvline(0.5, color="grey", ls="--", lw=1)
axes[1].set_xlim(0.45, 1.0)
axes[1].set_xlabel("ROC-AUC using this feature alone")
axes[1].set_title("B. Single-feature fraud signal (train)")
axes[1].legend(handles=[Patch(color="#C44E52", label="higher → more fraud"),
                        Patch(color="#4C72B0", label="lower → more fraud")], loc="lower right", fontsize=8)

if best_cat:
    rates = cat_rates[best_cat]
    axes[2].bar(rates.index.astype(str), rates["fraud_rate"], color="#DD8452")
    axes[2].axhline(y_train.mean(), color="black", ls="--", lw=1, label="Overall fraud rate")
    axes[2].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[2].tick_params(axis="x", rotation=30)
    axes[2].set_title(f"C. Fraud rate by {best_cat} (train)")
    axes[2].legend(fontsize=8)

plt.tight_layout()
save_fig("fig1_key_eda_findings")
plt.show()

In [ ]:
LEAKAGE_AUC = 0.95
suspects = uni_df.loc[uni_df["strength"] > LEAKAGE_AUC, "feature"].tolist()
weak = (uni_df["strength"] < 0.55).sum()
top3 = uni_df.head(3)

lines = [
    "#### Interpretation — what the data told us, and what we did about it",
    f"- **Imbalance:** {pct(y_all.mean())} of transactions are fraud ({int(y_all.sum()):,} of {len(y_all):,}). "
    f"A model that never flags fraud would still be {pct(1 - y_all.mean())} accurate, so we **ignore accuracy** and compare "
    "imbalance strategies (class weights vs SMOTE) using recall, precision, F1 and PR-AUC.",
    "- **Where the signal is:** the strongest single features are "
    + "; ".join(f"`{r.feature}` (AUC {r.strength:.2f}, {r.direction})" for r in top3.itertuples())
    + f". {weak} of {len(uni_df)} numeric features show almost no signal on their own (AUC < 0.55). "
      "Because fraud is spread over a few features, a regularized model that can **switch weak features off** (L1) is worth testing.",
    "- **Leakage check:** " + (f"⚠️ {suspects} separate fraud almost perfectly on their own — they must be checked for "
                               "post-event information before deployment." if suspects else
                               f"no feature exceeds AUC {LEAKAGE_AUC} on its own, so we found no sign of a target-leaking column."),
    f"- **Data quality:** {int(df.isnull().sum().sum())} missing cells and {int(df.duplicated().sum())} duplicate rows. "
    "We still put a median/most-frequent imputer inside the pipeline so the model will not break if future data has gaps.",
]
for c, r in cat_rates.items():
    lines.append(f"- **`{c}`:** the fraud rate ranges from {pct(r['fraud_rate'].min())} (`{r.index[-1]}`) "
                 f"to {pct(r['fraud_rate'].max())} (`{r.index[0]}`) against {pct(y_train.mean())} overall. "
                 "Milestone 4 excluded this field, so we now **test** whether adding it (one-hot encoded) improves the model.")
md("\n".join(lines))

## 4. Leakage-safe modelling pipeline

Every candidate is a single `imblearn` pipeline: **imputation → scaling (+ one-hot encoding when categorical fields are used) → optional SMOTE → classifier.** Because everything is fitted inside the pipeline on training rows only:

- the scaler never sees validation or test statistics (preprocessing leakage);
- SMOTE creates synthetic fraud **only from training rows** and is skipped automatically at prediction time (oversampling leakage);
- cross-validation refits every step inside each fold (cross-validation leakage).

In [ ]:
def _one_hot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:                                   # older scikit-learn
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(use_categorical=False):
    transformers = [("num", sk_make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_cols)]
    if use_categorical and categorical_cols:
        transformers.append(("cat", sk_make_pipeline(SimpleImputer(strategy="most_frequent"), _one_hot()),
                             categorical_cols))
    return ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)

def make_model(estimator, use_categorical=False, use_smote=False):
    steps = [("prep", make_preprocessor(use_categorical))]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=3)))
    steps.append(("model", estimator))
    return Pipeline(steps)

def metrics_at(y_true, scores, threshold=0.5):
    """Threshold-dependent metrics plus confusion counts."""
    pred = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(np.asarray(y_true), pred, labels=[0, 1]).ravel()
    return {"Precision": tp / (tp + fp) if tp + fp else 0.0,
            "Recall": tp / (tp + fn) if tp + fn else 0.0,
            "F1": 2 * tp / (2 * tp + fp + fn) if tp else 0.0,
            "FPR": fp / (fp + tn) if fp + tn else 0.0,
            "Alert rate": (tp + fp) / len(pred),
            "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}

def evaluate(pipe, X_part, y_part, threshold=0.5):
    scores = pipe.predict_proba(X_part)[:, 1]
    out = {"PR-AUC": average_precision_score(y_part, scores), "ROC-AUC": roc_auc_score(y_part, scores)}
    out.update(metrics_at(y_part, scores, threshold))
    return out

## 5. Candidate models and final model selection (Part 1, Q3)

| Candidate | Why we tested it |
|---|---|
| Baseline LR | Milestone 4 reference point, no imbalance handling |
| Balanced LR | Cost-sensitive learning: errors on fraud weigh more |
| SMOTE + LR | Synthetic oversampling (Milestone 4 augmentation question) |
| Tuned balanced LR | `C` (regularization strength) and L1/L2 penalty tuned on validation |
| Tuned balanced LR + categorical | Adds the payment-channel / authentication fields that Milestone 4 left out |
| Random forest (benchmark) | Non-linear reference: shows how much a linear model might be missing |

**Selection rule (decided before looking at results).** We rank candidates by **validation PR-AUC**. In Milestone 4 we used F1 at the default 0.5 threshold, but that unfairly penalizes unweighted models whose scores sit on a different scale. Since Milestone 5 chooses the decision threshold separately (§6), the fair comparison is **ranking quality**, and PR-AUC is the ranking metric that focuses on the rare fraud class. If a simpler model is within **0.01 PR-AUC** of the best, we prefer the simpler one: a bank must be able to explain an alert to analysts, auditors and customers. Complexity order: plain/weighted LR (1) < SMOTE or extra categorical inputs (2) < random forest (3).

In [ ]:
candidates = {}

def add_candidate(name, pipe, use_categorical, complexity, note, fit=True):
    if fit:
        pipe.fit(X_train, y_train)
    candidates[name] = {"pipe": pipe, "use_categorical": use_categorical,
                        "complexity": complexity, "note": note}

def tune_on_validation(builder, grid):
    """Fit each setting on training rows; keep the best validation PR-AUC."""
    rows, best = [], None
    for params in grid:
        pipe = builder(**params).fit(X_train, y_train)
        s = evaluate(pipe, X_val, y_val)
        rows.append({**params, **{k: s[k] for k in ["PR-AUC", "ROC-AUC", "F1", "Recall", "Precision"]}})
        if best is None or s["PR-AUC"] > best[1]:
            best = (pipe, s["PR-AUC"], params)
    return best, pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)

def lr(**kw):
    return LogisticRegression(max_iter=3000, random_state=RANDOM_STATE, **kw)

add_candidate("Baseline LR", make_model(lr()), False, 1, "No imbalance handling")
add_candidate("Balanced LR", make_model(lr(class_weight="balanced")), False, 1, "class_weight='balanced'")
add_candidate("SMOTE + LR", make_model(lr(), use_smote=True), False, 2, "SMOTE on training rows only")

lr_grid = [{"C": c, "penalty": p} for c in [0.01, 0.1, 1.0, 10.0] for p in ["l1", "l2"]]
def tuned_lr_builder(use_categorical):
    return lambda C, penalty: make_model(
        lr(C=C, penalty=penalty, solver="liblinear", class_weight="balanced"), use_categorical=use_categorical)

(best_pipe, _, p), tuning_numeric = tune_on_validation(tuned_lr_builder(False), lr_grid)
add_candidate("Tuned balanced LR", best_pipe, False, 1, f"C={p['C']}, penalty={p['penalty']}", fit=False)
print("Tuning grid — balanced LR, numeric features (top 5 by validation PR-AUC)")
display(tuning_numeric.head().round(3))

if categorical_cols:
    (best_pipe, _, p), tuning_cat = tune_on_validation(tuned_lr_builder(True), lr_grid)
    add_candidate("Tuned balanced LR + categorical", best_pipe, True, 2,
                  f"C={p['C']}, penalty={p['penalty']}, + one-hot {categorical_cols}", fit=False)
    print("Tuning grid — balanced LR + categorical features (top 5)")
    display(tuning_cat.head().round(3))

rf_grid = [{"max_depth": d, "min_samples_leaf": l} for d in [6, 12, None] for l in [1, 5]]
rf_builder = lambda max_depth, min_samples_leaf: make_model(
    RandomForestClassifier(n_estimators=300, max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                           class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    use_categorical=bool(categorical_cols))
(best_pipe, _, p), tuning_rf = tune_on_validation(rf_builder, rf_grid)
add_candidate("Random forest (benchmark)", best_pipe, bool(categorical_cols), 3,
              f"max_depth={p['max_depth']}, min_samples_leaf={p['min_samples_leaf']}", fit=False)
print("All candidates fitted on training rows:", list(candidates))

In [ ]:
val_rows = []
for name, c in candidates.items():
    s = evaluate(c["pipe"], X_val, y_val)
    val_rows.append({"Model": name, "PR-AUC": s["PR-AUC"], "ROC-AUC": s["ROC-AUC"],
                     "F1 @0.5": s["F1"], "Recall @0.5": s["Recall"], "Precision @0.5": s["Precision"],
                     "Complexity": c["complexity"], "Settings": c["note"]})
val_table = pd.DataFrame(val_rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)

SELECTION_TOLERANCE = 0.01
best_ap = val_table["PR-AUC"].max()
eligible = val_table[val_table["PR-AUC"] >= best_ap - SELECTION_TOLERANCE]
final_name = eligible.sort_values(["Complexity", "PR-AUC"], ascending=[True, False]).iloc[0]["Model"]
final = candidates[final_name]
final_pipe = final["pipe"]
FINAL_COLS = numeric_cols + (categorical_cols if final["use_categorical"] else [])

display(val_table.round(3))
print("Selected final model:", final_name, "|", final["note"])

### Figure 2 — Validation comparison of all candidates

**What it shows.** Validation PR-AUC and ROC-AUC (threshold-free ranking quality) and F1 at the default 0.5 threshold for every candidate. The selected model is marked with ★.

**Why it matters.** This is the evidence behind our model choice. It answers three questions from Milestone 4 directly: does handling imbalance help, is SMOTE better than class weights, and does a more complex model earn its loss of explainability?

In [ ]:
plot_df = val_table.melt(id_vars="Model", value_vars=["PR-AUC", "ROC-AUC", "F1 @0.5"],
                         var_name="Metric", value_name="Score")
plot_df["Model"] = plot_df["Model"].apply(lambda m: f"★ {m}" if m == final_name else m)
order = [f"★ {m}" if m == final_name else m for m in val_table["Model"]]

plt.figure(figsize=(11, 5.2))
ax = sns.barplot(data=plot_df, y="Model", x="Score", hue="Metric", order=order,
                 palette=["#C44E52", "#4C72B0", "#55A868"])
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.2f", fontsize=8, padding=2)
ax.axvline(y_val.mean(), color="grey", ls=":", lw=1.5, label=f"PR-AUC of random guessing ({y_val.mean():.3f})")
plt.xlim(0, 1.08)
plt.title("Validation performance of candidate models (★ = selected)")
plt.xlabel("Score"); plt.ylabel("")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
save_fig("fig2_model_comparison_validation")
plt.show()

In [ ]:
v = val_table.set_index("Model")["PR-AUC"]
base, bal, smo = v["Baseline LR"], v["Balanced LR"], v["SMOTE + LR"]
rf = v["Random forest (benchmark)"]
fin = v[final_name]
cat_line = ""
if "Tuned balanced LR + categorical" in v:
    gain = v["Tuned balanced LR + categorical"] - v["Tuned balanced LR"]
    cat_line = (f"- **Categorical fields:** adding the one-hot encoded fields changed validation PR-AUC by {gain:+.3f}. "
                + ("That is a real gain, so the excluded fields carried useful information."
                   if gain > SELECTION_TOLERANCE else
                   "That is within our 0.01 tolerance, so the extra inputs are not worth the added complexity."))

md(f"""#### Interpretation — why **{final_name}** ({final['note']})
- **Imbalance handling:** class weights changed PR-AUC from {base:.3f} (baseline) to {bal:.3f}, and SMOTE gave {smo:.3f}. \
{"Rebalancing mostly **shifts the score scale** (compare recall at 0.5 in the table) rather than improving ranking quality; " if abs(bal - base) < SELECTION_TOLERANCE else "Rebalancing changed ranking quality noticeably, not only the score scale; "}\
this is why we choose the decision threshold separately in §6 instead of relying on 0.5.
- **SMOTE vs class weights:** SMOTE {'did not beat' if smo <= bal + SELECTION_TOLERANCE else 'beat'} simple class weights \
({smo:.3f} vs {bal:.3f}). Class weights achieve the same goal without inventing synthetic transactions, avoiding the \
synthetic-data risks we listed in Milestone 4.
{cat_line}
- **Non-linear benchmark:** the random forest reached {rf:.3f} vs {fin:.3f} for our selected model. \
{"The gap is small, so a linear model captures most of the available signal and keeps full explainability." if rf - fin <= 0.03 else "The forest finds patterns a linear model misses; this is a clear direction for future work."}
- **Decision:** best validation PR-AUC was {best_ap:.3f}; **{final_name}** scored {fin:.3f} and is the simplest model within \
{SELECTION_TOLERANCE} of the best. The test set has not been used for any of these choices.
""")

## 6. Choosing the decision threshold (a business decision, made on validation data)

A model outputs a risk score; the bank decides above which score a transaction is sent for review. The default 0.5 is arbitrary — especially after class weighting, which inflates scores. We therefore choose the threshold on **validation** data with two policies:

- **F1-optimal threshold** (our operating point): the best balance between catching fraud and avoiding false alerts.
- **Recall-target threshold**: the highest threshold that still catches at least 90% of validation fraud — for a bank that prioritizes not missing fraud.

### Figure 3 — Precision, recall and alert volume across thresholds

**What it shows.** How precision, recall, F1 and the share of transactions flagged change as the threshold moves, on validation rows. **Why it matters.** It makes the trade-off visible: every extra fraud caught costs some extra false alerts. The bank can pick a point that fits its review capacity.

In [ ]:
val_scores = final_pipe.predict_proba(X_val)[:, 1]
test_scores = final_pipe.predict_proba(X_test)[:, 1]
train_scores = final_pipe.predict_proba(X_train)[:, 1]

grid = np.round(np.linspace(0.02, 0.98, 97), 2)
thr_df = pd.DataFrame([{"threshold": t, **metrics_at(y_val, val_scores, t)} for t in grid])

RECALL_TARGET = 0.90
t_f1 = float(thr_df.loc[thr_df["F1"].idxmax(), "threshold"])
ok = thr_df[thr_df["Recall"] >= RECALL_TARGET]
t_recall = float(ok["threshold"].max()) if len(ok) else float(grid.min())
OPERATING_THRESHOLD = t_f1

fig, ax = plt.subplots(figsize=(10, 5))
prec_plot = thr_df["Precision"].where(thr_df["TP"] + thr_df["FP"] > 0)   # undefined when nothing is flagged
ax.plot(thr_df["threshold"], prec_plot, label="Precision", color="#4C72B0", lw=2)
ax.plot(thr_df["threshold"], thr_df["Recall"], label="Recall", color="#C44E52", lw=2)
ax.plot(thr_df["threshold"], thr_df["F1"], label="F1", color="#55A868", lw=2.5)
ax.plot(thr_df["threshold"], thr_df["Alert rate"], label="Share of transactions flagged", color="grey", ls="--")
for t, lab, col in [(0.5, "default 0.50", "black"), (t_f1, f"F1-optimal {t_f1:.2f}", "#55A868"),
                    (t_recall, f"≥{RECALL_TARGET:.0%} recall {t_recall:.2f}", "#C44E52")]:
    ax.axvline(t, color=col, ls=":", lw=1.8, label=f"Threshold: {lab}")
ax.set_ylim(0, 1.05); ax.set_xlim(0, 1)
ax.set_xlabel("Decision threshold on fraud score"); ax.set_ylabel("Validation value")
ax.set_title(f"Threshold trade-off on validation data — {final_name}")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)
plt.tight_layout()
save_fig("fig3_threshold_tradeoff")
plt.show()

thr_compare = pd.DataFrame({name: metrics_at(y_val, val_scores, t) for name, t in
                            [("Default 0.50", 0.5), (f"F1-optimal {t_f1:.2f}", t_f1),
                             (f"Recall ≥{RECALL_TARGET:.0%} {t_recall:.2f}", t_recall)]}).T
display(thr_compare[["Precision", "Recall", "F1", "Alert rate", "TP", "FP", "FN"]].round(3))

In [ ]:
d, o = thr_compare.iloc[0], thr_compare.iloc[1]
md(f"""#### Interpretation
- At the default 0.50, the model flags {pct(d['Alert rate'])} of validation transactions with precision {pct(d['Precision'])} and recall {pct(d['Recall'])}.
- At the **F1-optimal threshold {t_f1:.2f}** it flags {pct(o['Alert rate'])}, with precision {pct(o['Precision'])} and recall {pct(o['Recall'])} (F1 {o['F1']:.3f} vs {d['F1']:.3f} at 0.50).
- Holding recall at {RECALL_TARGET:.0%} needs threshold {t_recall:.2f}, which raises the alert share to {pct(thr_compare.iloc[2]['Alert rate'])}.

**Connection to the problem.** The threshold, not the model, decides how many customers get stopped and how busy analysts are. \
We adopt **{OPERATING_THRESHOLD:.2f}** as the operating point and apply it unchanged to the test set below; §10 shows how the choice moves under different cost assumptions.
""")

## 7. Final model performance (Part 1, Q4)

The model and threshold are now frozen. We open the test set once and compare training, validation and test results. Similar numbers across the three parts mean the model generalizes; a much higher training score would signal overfitting.

In [ ]:
rows = []
for part, Xp, yp in [("Training", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    for label, t in [("0.50 (default)", 0.5), (f"{OPERATING_THRESHOLD:.2f} (operating)", OPERATING_THRESHOLD)]:
        s = evaluate(final_pipe, Xp, yp, t)
        rows.append({"Data part": part, "Threshold": label,
                     **{k: s[k] for k in ["PR-AUC", "ROC-AUC", "Precision", "Recall", "F1", "FPR", "TP", "FP", "FN", "TN"]}})
final_scores = pd.DataFrame(rows)
display(final_scores.round(3))

test_op = evaluate(final_pipe, X_test, y_test, OPERATING_THRESHOLD)
test_def = evaluate(final_pipe, X_test, y_test, 0.5)
train_op = evaluate(final_pipe, X_train, y_train, OPERATING_THRESHOLD)
val_op = evaluate(final_pipe, X_val, y_val, OPERATING_THRESHOLD)
gap = train_op["PR-AUC"] - test_op["PR-AUC"]

md(f"""#### Interpretation — headline test result for **{final_name}** at threshold {OPERATING_THRESHOLD:.2f}
- The model caught **{test_op['TP']} of {test_op['TP'] + test_op['FN']} fraud cases (recall {pct(test_op['Recall'])})** in the unseen test set, \
and **{pct(test_op['Precision'])} of its alerts were real fraud** (about 1 in {1 / max(test_op['Precision'], 1e-9):.1f} alerts). F1 = {test_op['F1']:.3f}, \
PR-AUC = {test_op['PR-AUC']:.3f} (random guessing would score {y_test.mean():.3f}), ROC-AUC = {test_op['ROC-AUC']:.3f}.
- {pct(test_op['FPR'])} of legitimate transactions were flagged (false-positive rate), so {pct(1 - test_op['FPR'])} of good customers pass without friction.
- **Generalization:** PR-AUC is {train_op['PR-AUC']:.3f} on training, {val_op['PR-AUC']:.3f} on validation and {test_op['PR-AUC']:.3f} on test (train–test gap {gap:+.3f}). \
{"The small gap means the model is not memorizing training rows." if abs(gap) < 0.05 else "The gap is noticeable, so some overfitting is likely and the regularization should be revisited."} \
Test F1 ({test_op['F1']:.3f}) vs validation F1 ({val_op['F1']:.3f}) confirms that the validation-chosen threshold carries over to new rows.
""")

### Figure 4 — Confusion matrices on the test set

**What it shows.** The four outcomes on unseen test transactions at the default 0.50 threshold and at our operating threshold. Each cell shows the count and its share of the actual class. **Why it matters.** It translates abstract metrics into the two business errors: frauds that slip through (bottom-left) and customers wrongly flagged (top-right).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, (title, t) in zip(axes, [("Default threshold 0.50", 0.5), (f"Operating threshold {OPERATING_THRESHOLD:.2f}", OPERATING_THRESHOLD)]):
    cm = confusion_matrix(y_test, (test_scores >= t).astype(int), labels=[0, 1])
    row_share = cm / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f"{cm[i, j]:,}\n({row_share[i, j]:.1%})" for j in range(2)] for i in range(2)])
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=ax, annot_kws={"size": 13},
                xticklabels=["Predicted: not fraud", "Predicted: fraud"],
                yticklabels=["Actual: not fraud", "Actual: fraud"])
    ax.set_title(title)
fig.suptitle(f"Test-set confusion matrices — {final_name}", fontsize=13)
plt.tight_layout()
save_fig("fig4_confusion_matrices_test")
plt.show()

md(f"""#### Interpretation
- Moving from 0.50 to {OPERATING_THRESHOLD:.2f} changes missed frauds from **{test_def['FN']} to {test_op['FN']}** and false alerts from \
**{test_def['FP']} to {test_op['FP']}** on {len(y_test):,} test transactions.
- **Connection to the problem:** each false negative is a loss the bank absorbs; each false positive is a customer interruption and an analyst task. \
The matrix gives the review team a concrete expectation: out of every 1,000 transactions like these, about \
{1000 * (test_op['TP'] + test_op['FP']) / len(y_test):.0f} will be flagged and {1000 * test_op['TP'] / len(y_test):.0f} of those will be fraud.
""")

### Figure 5 — ROC and Precision–Recall curves on the test set

**What it shows.** The ROC curve plots fraud caught (recall) against legitimate transactions wrongly flagged at every threshold; the PR curve plots precision against recall. We compare the final model with the Milestone 4 baseline and the SMOTE model; the dot marks our operating point. **Why it matters.** With only ~12.5% fraud, the PR curve is the more honest view: ROC can look strong even when many alerts are false, while PR shows directly how clean the alert queue is.

In [ ]:
curve_models = {final_name: final_pipe}
for name in ["Baseline LR", "SMOTE + LR"]:
    if name != final_name:
        curve_models[name] = candidates[name]["pipe"]
palette = ["#C44E52", "#4C72B0", "#8172B3", "#55A868"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for (name, pipe), col in zip(curve_models.items(), palette):
    s = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, s)
    prec, rec, _ = precision_recall_curve(y_test, s)
    lw = 3 if name == final_name else 1.6
    axes[0].plot(fpr, tpr, color=col, lw=lw, label=f"{name} (AUC {roc_auc_score(y_test, s):.3f})")
    axes[1].plot(rec, prec, color=col, lw=lw, label=f"{name} (AP {average_precision_score(y_test, s):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[0].scatter(test_op["FPR"], test_op["Recall"], s=90, color="black", zorder=5, label=f"Operating point ({OPERATING_THRESHOLD:.2f})")
axes[0].set_xlabel("False positive rate (legitimate flagged)"); axes[0].set_ylabel("True positive rate (fraud caught)")
axes[0].set_title("ROC curve — test set")
axes[1].axhline(y_test.mean(), color="k", ls="--", lw=1, label=f"Random (fraud share {y_test.mean():.3f})")
axes[1].scatter(test_op["Recall"], test_op["Precision"], s=90, color="black", zorder=5, label="Operating point")
axes[1].set_xlabel("Recall (fraud caught)"); axes[1].set_ylabel("Precision (alerts that are fraud)")
axes[1].set_title("Precision–Recall curve — test set")
for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.legend(loc="lower right" if ax is axes[0] else "lower left", fontsize=8)
plt.tight_layout()
save_fig("fig5_roc_pr_curves_test")
plt.show()

base_ap = average_precision_score(y_test, candidates["Baseline LR"]["pipe"].predict_proba(X_test)[:, 1])
test_ap = {n: average_precision_score(y_test, c["pipe"].predict_proba(X_test)[:, 1]) for n, c in candidates.items()}
val_ap = val_table.set_index("Model")["PR-AUC"].to_dict()
beats_test = [n for n in curve_models if n != final_name and test_ap[n] > test_ap[final_name]]
if not beats_test:
    order_line = "On the test set the final model is at least as good as the baseline and SMOTE models, so the validation-based choice held up."
else:
    order_line = ("On the test set " + ", ".join(f"{n} ({test_ap[n]:.3f})" for n in beats_test)
                  + f" scored slightly above the final model ({test_ap[final_name]:.3f}). "
                  + ("The differences are within the cross-validation spread in §7, so they are noise rather than a reason to change the model — "
                     "and we do **not** re-select on test data." if max(test_ap[n] for n in beats_test) - test_ap[final_name] < 0.02 else
                     "This difference is larger than expected; we report it honestly but do not re-select on test data, as that would leak test information."))
md(f"""#### Interpretation
- On the test set the final model's PR-AUC is **{test_op['PR-AUC']:.3f}**, against {base_ap:.3f} for the Milestone 4 baseline and {y_test.mean():.3f} for random guessing — \
roughly **{test_op['PR-AUC'] / y_test.mean():.1f}× better than chance** at putting fraud at the top of the queue.
- {order_line}
- **Connection:** the PR curve is the menu the bank chooses from. Moving right along the curve catches more fraud but makes the alert queue less clean; \
our operating point sits where F1 is highest on validation data.
""")

### Figure 6 — Score separation and calibration

**What it shows.** (Left) The distribution of fraud scores for real fraud vs legitimate test transactions, with the operating threshold. (Right) A calibration curve: for transactions given, say, a 0.6 score, what share was actually fraud? **Why it matters.** Clear separation means confident decisions. Calibration tells us whether the score can be read as a **probability** — important if the bank wants to say "this transaction has a 70% chance of fraud" or to price risk. Class weighting and SMOTE are known to push scores upward, which we flagged as a risk in Milestone 4.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.histplot(x=test_scores, hue=y_test.map({0: "Legitimate", 1: "Fraud"}).values, bins=40, stat="density",
             common_norm=False, element="step", palette={"Legitimate": "#4C72B0", "Fraud": "#C44E52"}, ax=axes[0])
axes[0].axvline(OPERATING_THRESHOLD, color="black", ls="--", label=f"Threshold {OPERATING_THRESHOLD:.2f}")
axes[0].set_xlabel("Fraud score"); axes[0].set_title("Score distribution by actual class (test)")

for name, col in [(final_name, "#C44E52"), ("Baseline LR", "#4C72B0")]:
    s = candidates[name]["pipe"].predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, s, n_bins=10, strategy="quantile")
    axes[1].plot(mean_pred, frac_pos, "o-", color=col, label=f"{name} (Brier {brier_score_loss(y_test, s):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
axes[1].set_xlabel("Average predicted score"); axes[1].set_ylabel("Actual fraud share")
axes[1].set_title("Calibration curve (test)"); axes[1].legend(fontsize=8)
plt.tight_layout()
save_fig("fig6_score_distribution_calibration")
plt.show()

frac_pos, mean_pred = calibration_curve(y_test, test_scores, n_bins=10, strategy="quantile")
over = np.mean(mean_pred - frac_pos)
md(f"""#### Interpretation
- Median score is {np.median(test_scores[y_test.values == 1]):.2f} for fraud vs {np.median(test_scores[y_test.values == 0]):.2f} for legitimate transactions; \
the overlap region around the threshold is where the errors in Figure 4 come from.
- On average the final model's scores are {abs(over):.2f} {'above' if over > 0 else 'below'} the true fraud rate in each bin. \
{"Scores are inflated, as expected from class weighting, so they should be treated as a **risk ranking**, not as literal probabilities." if over > 0.05 else "Scores are reasonably close to real probabilities."}
- **Connection:** ranking and thresholding (what we need for an alert queue) are unaffected, but if the bank wants probabilities, \
the model should be recalibrated (e.g. `CalibratedClassifierCV`) on fresh data.
""")

### Figure 7 — Robustness: repeated cross-validation

**What it shows.** PR-AUC and ROC-AUC of the final model configuration and the baseline across 15 different train/validation splits of the development data (5-fold stratified CV repeated 3 times; the test set stays out). Every pipeline step, including scaling, is refitted inside each fold. **Why it matters.** One split can be lucky. A narrow spread means the result is a property of the method, not of which rows happened to land in validation. This replaces the three-split sensitivity check from Milestone 4.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
cv_rows = []
for name in dict.fromkeys([final_name, "Baseline LR"]):
    cols = numeric_cols + (categorical_cols if candidates[name]["use_categorical"] else [])
    res = cross_validate(clone(candidates[name]["pipe"]), X_train_val[cols], y_train_val, cv=cv,
                         scoring={"PR-AUC": "average_precision", "ROC-AUC": "roc_auc"}, n_jobs=-1)
    for m in ["PR-AUC", "ROC-AUC"]:
        cv_rows += [{"Model": name, "Metric": m, "Score": v} for v in res[f"test_{m}"]]
cv_df = pd.DataFrame(cv_rows)

plt.figure(figsize=(9, 4.5))
sns.boxplot(data=cv_df, x="Metric", y="Score", hue="Model", palette=["#C44E52", "#4C72B0"])
sns.stripplot(data=cv_df, x="Metric", y="Score", hue="Model", dodge=True, color="black", size=3, legend=False)
plt.title("Repeated 5-fold CV on development data (15 fits per model)")
plt.tight_layout()
save_fig("fig7_cv_robustness")
plt.show()

cv_summary = cv_df.groupby(["Model", "Metric"])["Score"].agg(["mean", "std"]).round(3)
display(cv_summary)
fm, fs = cv_summary.loc[(final_name, "PR-AUC")]
md(f"""#### Interpretation
- Across 15 refits, the final configuration's PR-AUC is **{fm:.3f} ± {fs:.3f}** (mean ± standard deviation); the single test-set value was {test_op['PR-AUC']:.3f}.
- {"The test result sits inside the cross-validation range, so it is not a lucky split." if abs(test_op['PR-AUC'] - fm) <= 2 * fs + 0.01 else "The test result falls outside the usual CV range, so we treat the single test number with caution."}
- **Connection:** a bank needs a model whose performance is predictable. {"The low spread supports deploying this configuration in a monitored pilot." if fs < 0.03 else "The spread is wide, so more data or a simpler model is needed before relying on a single number."}
""")

## 8. What did the model learn? Explainability (Part 1, Q5 · Part 2)

We use three complementary methods so that no single technique drives our conclusions:

1. **SHAP values** — how much each feature pushed each individual prediction up or down (global summary + individual cases).
2. **Model coefficients as odds ratios** — the native explanation for logistic regression.
3. **Permutation importance** — how much test PR-AUC drops when a feature is shuffled; model-agnostic and measured on unseen data.

If the three agree, we can trust the story about which features drive fraud alerts.

In [ ]:
prep = final_pipe.named_steps["prep"]
model = final_pipe.named_steps["model"]
feat_names = list(prep.get_feature_names_out())
Xtr_t = pd.DataFrame(prep.transform(X_train), columns=feat_names, index=X_train.index)
Xte_t = pd.DataFrame(prep.transform(X_test), columns=feat_names, index=X_test.index)

sv = None
if HAS_SHAP:
    background = Xtr_t.sample(min(100, len(Xtr_t)), random_state=RANDOM_STATE)
    if isinstance(model, LogisticRegression):
        explainer = shap.LinearExplainer(model, background)
    elif isinstance(model, RandomForestClassifier):
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.Explainer(lambda a: model.predict_proba(a)[:, 1], background)
    sv = explainer(Xte_t)
    if sv.values.ndim == 3:            # tree models return one set of values per class
        sv = sv[:, :, 1]
    # Show original units (not z-scores) in every SHAP plot; one-hot columns stay 0/1
    raw_display = np.column_stack([X_test[f].astype(float).values if f in numeric_cols else Xte_t[f].values
                                   for f in feat_names])
    sv = shap.Explanation(values=sv.values, base_values=sv.base_values, data=raw_display, feature_names=feat_names)
    shap_importance = pd.Series(np.abs(sv.values).mean(axis=0), index=feat_names).sort_values(ascending=False)
    print("SHAP values computed for", sv.values.shape[0], "test transactions.",
          "Units: log-odds of fraud." if isinstance(model, LogisticRegression) else "")

### Figure 8 — SHAP summary (global explanation)

**What it shows.** Each dot is one test transaction. Its horizontal position is how much that feature pushed the fraud score up (right) or down (left); its colour is the feature value (red = high, blue = low). Features are ordered by overall impact. For logistic regression the SHAP values are in log-odds units, and they add up exactly to the model's score for each transaction. **Why it matters.** It tells analysts *which* signals the model relies on and *in which direction* — needed to trust the model, to brief investigators, and to spot features that behave suspiciously.

In [ ]:
if sv is not None:
    fig = plt.figure()
    shap.plots.beeswarm(sv, max_display=12, show=False)
    plt.title(f"SHAP summary — {final_name} (test set)")
    plt.gcf().set_size_inches(10, 6)
    plt.tight_layout()
    save_fig("fig8_shap_summary", plt.gcf())
    plt.show()

    corr_dir = {f: np.corrcoef(Xte_t[f], sv[:, f].values)[0, 1] if Xte_t[f].std() > 0 else 0 for f in shap_importance.index[:5]}
    total = shap_importance.sum()
    md("#### Interpretation\n" + "\n".join(
        f"- **`{f}`** — {pct(shap_importance[f] / total)} of total SHAP impact; "
        + ("higher values **raise** the fraud score." if corr_dir[f] > 0 else "higher values **lower** the fraud score.")
        for f in shap_importance.index[:5])
        + f"\n\nThe top 5 features account for {pct(shap_importance.iloc[:5].sum() / total)} of the model's total attribution. "
          "**Connection:** these are the signals investigators should check first when an alert arrives, and the fields "
          "whose data quality matters most in production.")
else:
    print("SHAP not available — see coefficient and permutation importance plots below.")

### Figure 9 — Coefficients (odds ratios) and permutation importance

**What it shows.** (Left, logistic regression only) How the odds of fraud multiply when a feature rises by one standard deviation, holding the others fixed; bars right of 1 raise risk, left of 1 lower it, and features shrunk to exactly zero by L1 regularization are listed below. (Right) How much test PR-AUC falls when each raw input is randomly shuffled, with error bars over 10 shuffles. **Why it matters.** Coefficients give a precise, auditable rule-of-thumb; permutation importance checks on unseen data that the features the model *uses* are also the ones that *help*.

In [ ]:
perm = permutation_importance(final_pipe, X_test[FINAL_COLS], y_test, scoring="average_precision",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_df = (pd.DataFrame({"feature": FINAL_COLS, "drop": perm.importances_mean, "std": perm.importances_std})
           .sort_values("drop", ascending=False).reset_index(drop=True))

is_lr = isinstance(model, LogisticRegression)
fig, axes = plt.subplots(1, 2 if is_lr else 1, figsize=(14 if is_lr else 7, 5.5))
axes = np.atleast_1d(axes)
zeroed = []
if is_lr:
    coefs = pd.Series(model.coef_[0], index=feat_names)
    zeroed = coefs[coefs == 0].index.tolist()
    top = coefs[coefs != 0].reindex(coefs[coefs != 0].abs().sort_values(ascending=False).index).head(12).iloc[::-1]
    odds = np.exp(top)
    axes[0].barh(top.index, odds - 1, left=1, color=["#C44E52" if v > 0 else "#4C72B0" for v in top])
    axes[0].axvline(1, color="black", lw=1)
    axes[0].set_xlabel("Odds ratio for +1 standard deviation (1 = no effect)")
    axes[0].set_title("Logistic regression odds ratios (non-zero, top 12)")
ax = axes[-1]
top_perm = perm_df.head(12).iloc[::-1]
ax.barh(top_perm["feature"], top_perm["drop"], xerr=top_perm["std"], color="#8172B3")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Drop in test PR-AUC when shuffled")
ax.set_title("Permutation importance (test set)")
plt.tight_layout()
save_fig("fig9_coefficients_permutation")
plt.show()

lines = ["#### Interpretation"]
if is_lr:
    best_up = coefs.idxmax(); best_down = coefs.idxmin()
    lines.append(f"- The strongest risk-raising feature is **`{best_up}`**: one standard deviation higher multiplies the odds of fraud by **{np.exp(coefs[best_up]):.2f}×**. "
                 f"The strongest protective feature is **`{best_down}`** (odds × {np.exp(coefs[best_down]):.2f}).")
    if zeroed:
        lines.append(f"- L1 regularization set **{len(zeroed)} of {len(coefs)}** inputs to exactly zero ({', '.join('`'+z+'`' for z in zeroed[:8])}{'…' if len(zeroed) > 8 else ''}): "
                     "the model ignores them, which makes it simpler to explain and to maintain.")
useless = perm_df[perm_df["drop"] <= 0]["feature"].tolist()
lines.append(f"- Shuffling **`{perm_df.loc[0, 'feature']}`** cuts test PR-AUC by {perm_df.loc[0, 'drop']:.3f} — the single most valuable input. "
             f"{len(useless)} inputs cause no measurable drop when shuffled.")
if sv is not None:
    overlap = set(shap_importance.index[:5]) & set(perm_df["feature"].head(5))
    lines.append(f"- **Agreement check:** {len(overlap)} of the top-5 SHAP features are also in the top-5 permutation features "
                 f"({', '.join('`'+o+'`' for o in overlap) or 'none'}). "
                 + ("The methods agree, so the explanation is robust." if len(overlap) >= 3 else
                    "Partial agreement — correlated features can share credit differently across methods, so we read them together."))
lines.append("- **Connection:** these are the drivers the bank can turn into investigator guidance and monitor for drift (§11).")
md("\n".join(lines))

### Figure 10 — SHAP dependence: how the top features change risk

**What it shows.** For the two most influential features, each dot is a test transaction: the feature value (x-axis) against its SHAP contribution (y-axis), coloured by the feature it interacts with most. **Why it matters.** It shows the *shape* of each effect — for logistic regression a straight line by construction — and where the contribution crosses from lowering to raising risk, which can be turned into simple business rules for analysts.

In [ ]:
if sv is not None:
    for i, f in enumerate(shap_importance.index[:2]):
        shap.plots.scatter(sv[:, f], color=sv, show=False)
        plt.title(f"SHAP dependence — {f}")
        plt.gcf().set_size_inches(7.5, 4.5)
        plt.tight_layout()
        save_fig(f"fig10_shap_dependence_{i + 1}", plt.gcf())
        plt.show()
    f1_, f2_ = shap_importance.index[:2]
    def crossing(f):
        vals, contr = X_test[f].values if f in X_test else Xte_t[f].values, sv[:, f].values
        pos = vals[contr > 0]
        return f"contributions turn positive (risk-raising) from about **{np.min(pos):,.2f}**" if len(pos) and np.corrcoef(vals, contr)[0, 1] > 0 else \
               (f"contributions are positive (risk-raising) below about **{np.max(pos):,.2f}**" if len(pos) else "contributions stay negative")
    md(f"""#### Interpretation
- **`{f1_}`:** {crossing(f1_)} (original units).
- **`{f2_}`:** {crossing(f2_)} (original units).
- **Connection:** these cut-points give investigators an intuitive reading of an alert ("this one was flagged mainly because `{f1_}` was unusually {'high' if np.corrcoef(Xte_t[f1_], sv[:, f1_].values)[0, 1] > 0 else 'low'}"). \
Because a linear model can only draw straight lines in log-odds space, any curved effect in reality is a candidate for feature engineering or a non-linear model later.
""")

### Figure 11 — Individual explanations: a caught fraud, a missed fraud, a false alert

**What it shows.** SHAP waterfall plots for three test transactions: the most confidently caught fraud, the fraud the model was *most* sure was legitimate, and the legitimate transaction it was *most* sure was fraud. Each bar shows how one feature moved the score from the average starting value. **Why it matters.** This is how an investigator would see an explanation for a single alert, and it reveals *why* the model fails on its worst errors.

In [ ]:
pred_op = (test_scores >= OPERATING_THRESHOLD).astype(int)
yt = y_test.values
cases = {}
for label, mask, pick in [("Caught fraud (true positive)", (yt == 1) & (pred_op == 1), np.argmax),
                          ("Missed fraud (false negative)", (yt == 1) & (pred_op == 0), np.argmin),
                          ("False alert (false positive)", (yt == 0) & (pred_op == 1), np.argmax)]:
    idx = np.where(mask)[0]
    if len(idx):
        cases[label] = idx[pick(test_scores[idx])]

if sv is not None:
    for k, (label, pos) in enumerate(cases.items()):
        shap.plots.waterfall(sv[pos], max_display=10, show=False)
        plt.title(f"{label} — score {test_scores[pos]:.2f}")
        plt.gcf().set_size_inches(8, 5)
        plt.tight_layout()
        save_fig(f"fig11_shap_case_{k + 1}", plt.gcf())
        plt.show()
    lines = ["#### Interpretation"]
    for label, pos in cases.items():
        contrib = pd.Series(sv[pos].values, index=feat_names)
        up, down = contrib.idxmax(), contrib.idxmin()
        lines.append(f"- **{label}** (score {test_scores[pos]:.2f}): pushed up most by `{up}` ({contrib[up]:+.2f}), pulled down most by `{down}` ({contrib[down]:+.2f}).")
    fn_label = "Missed fraud (false negative)"
    if fn_label in cases:
        top5 = list(shap_importance.index[:5])
        fn_push = float(pd.Series(sv[cases[fn_label]].values, index=feat_names)[top5].sum())
        lines.append(f"- **Connection:** for the missed fraud, the five globally most important features together contribute {fn_push:+.2f} "
                     + ("— they point *away* from fraud, so this case looks legitimate on exactly the signals the model trusts most (*camouflaged fraud*). "
                        "Catching it needs information the model does not have, such as customer history, device or merchant patterns."
                        if fn_push < 0 else
                        "— they point slightly towards fraud, but not enough to cross the threshold; a lower threshold would catch it at the cost of more false alerts."))
    md("\n".join(lines))

## 9. Error analysis — where and for whom does the model fail?

### Figure 12 — Feature profile of each prediction outcome

**What it shows.** For the most important numeric features, the average value (in standard deviations from the training mean) of four groups of test transactions: caught frauds (TP), missed frauds (FN), false alerts (FP) and correctly passed legitimate transactions (TN). **Why it matters.** If missed frauds look like legitimate transactions, the model cannot be fixed by tuning — it needs new information. If false alerts look like fraud, the bank should expect some friction for a specific kind of customer.

In [ ]:
mu, sd = X_train[numeric_cols].mean(), X_train[numeric_cols].std().replace(0, 1)
z = (X_test[numeric_cols] - mu) / sd
outcome = np.select([(yt == 1) & (pred_op == 1), (yt == 1) & (pred_op == 0), (yt == 0) & (pred_op == 1)],
                    ["TP: caught fraud", "FN: missed fraud", "FP: false alert"], "TN: correct pass")
order = ["TP: caught fraud", "FN: missed fraud", "FP: false alert", "TN: correct pass"]
importance_order = (shap_importance.index if sv is not None else perm_df["feature"])
top_num = [f for f in importance_order if f in numeric_cols][:10]
profile = z.groupby(outcome).mean().T.reindex(index=top_num, columns=order)
group_n = pd.Series(outcome).value_counts().reindex(order).fillna(0).astype(int)

plt.figure(figsize=(10, 0.5 * len(top_num) + 2))
sns.heatmap(profile, annot=True, fmt=".2f", cmap="RdBu_r", center=0, cbar_kws={"label": "Mean z-score"})
plt.xticks(np.arange(len(order)) + 0.5, [f"{o}\n(n={group_n[o]})" for o in order], rotation=0, fontsize=9)
plt.title("Average feature values by prediction outcome (test, standardized)")
plt.tight_layout()
save_fig("fig12_error_profile")
plt.show()

lines = ["#### Interpretation"]
for f in top_num[:3]:
    tp_, fn_, tn_ = profile.loc[f, "TP: caught fraud"], profile.loc[f, "FN: missed fraud"], profile.loc[f, "TN: correct pass"]
    closer = "legitimate transactions" if abs(fn_ - tn_) < abs(fn_ - tp_) else "caught frauds"
    lines.append(f"- **`{f}`:** caught frauds average {tp_:+.2f} SD, missed frauds {fn_:+.2f} SD, legitimate {tn_:+.2f} SD — missed frauds look more like **{closer}**.")
camouflaged = sum(abs(profile.loc[f, "FN: missed fraud"] - profile.loc[f, "TN: correct pass"]) <
                  abs(profile.loc[f, "FN: missed fraud"] - profile.loc[f, "TP: caught fraud"]) for f in top_num[:3]) >= 2
lines.append(f"- {group_n['FN: missed fraud']} frauds were missed and {group_n['FP: false alert']} legitimate transactions were flagged at threshold {OPERATING_THRESHOLD:.2f}.")
lines.append("- **Connection:** " + ("missed frauds resemble legitimate activity on the top features, so a lower threshold would mostly add false alerts. "
                                     "The fix is richer features (history, velocity, device), not more tuning of this model." if camouflaged else
                                     "missed frauds still differ from legitimate activity on the top features, so a lower threshold or a non-linear model could recover some of them."))
md("\n".join(lines))

### Figure 13 — Segment analysis: is performance consistent across channels and authentication types?

**What it shows.** Recall (fraud caught) and false-positive rate (legitimate customers flagged) within each payment channel and authentication type on the test set. Segments with fewer than 20 fraud cases are hatched because their recall is unreliable. **Why it matters.** A model can look good overall while failing one group of customers. Uneven false-positive rates are a **fairness** concern (some customers are inconvenienced more), and uneven recall is a **security** gap (fraudsters will move to the weakest channel).

In [ ]:
seg_tables = []
if categorical_cols:
    fig, axes = plt.subplots(1, len(categorical_cols), figsize=(7 * len(categorical_cols), 4.5), squeeze=False)
    for ax, c in zip(axes[0], categorical_cols):
        rows = []
        for val, idx in X_test.groupby(c).groups.items():
            pos = X_test.index.get_indexer(idx)
            m = metrics_at(yt[pos], test_scores[pos], OPERATING_THRESHOLD)
            rows.append({"Field": c, "Segment": str(val), "Transactions": len(pos), "Frauds": int(yt[pos].sum()),
                         "Fraud rate": yt[pos].mean(), "Recall": m["Recall"], "FPR": m["FPR"], "Precision": m["Precision"]})
        seg = pd.DataFrame(rows).sort_values("Transactions", ascending=False)
        seg_tables.append(seg)
        xs = np.arange(len(seg))
        bars_r = ax.bar(xs - 0.2, seg["Recall"], 0.4, label="Recall", color="#C44E52")
        ax.bar(xs + 0.2, seg["FPR"], 0.4, label="False-positive rate", color="#4C72B0")
        for b, n in zip(bars_r, seg["Frauds"]):
            if n < 20:
                b.set_hatch("//"); b.set_alpha(0.6)
        ax.axhline(test_op["Recall"], color="#C44E52", ls="--", lw=1)
        ax.axhline(test_op["FPR"], color="#4C72B0", ls="--", lw=1)
        ax.set_xticks(xs); ax.set_xticklabels(seg["Segment"], rotation=30, ha="right")
        ax.set_ylim(0, 1.05); ax.set_title(f"Performance by {c} (test)")
        ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig("fig13_segment_analysis")
    plt.show()
    seg_all = pd.concat(seg_tables, ignore_index=True)
    display(seg_all.round(3))

    flags = []
    for r in seg_all.itertuples():
        if r.Frauds >= 20 and r.Recall < test_op["Recall"] - 0.10:
            flags.append(f"`{r.Field}={r.Segment}` recall {pct(r.Recall)} vs {pct(test_op['Recall'])} overall")
        if r.Transactions >= 100 and r.FPR > 1.5 * test_op["FPR"] and r.FPR - test_op["FPR"] > 0.02:
            flags.append(f"`{r.Field}={r.Segment}` false-positive rate {pct(r.FPR)} vs {pct(test_op['FPR'])} overall")
    used = "uses" if final["use_categorical"] else "does **not** use"
    md(f"""#### Interpretation
- Recall ranges from {pct(seg_all['Recall'].min())} to {pct(seg_all['Recall'].max())} and the false-positive rate from {pct(seg_all['FPR'].min())} to {pct(seg_all['FPR'].max())} across segments. The final model {used} these fields as inputs.
- {"**Segments needing attention:** " + "; ".join(flags) + "." if flags else "No segment with enough data falls more than 10 points below overall recall or has a clearly elevated false-positive rate."}
- **Connection:** before deployment the bank should set a maximum acceptable gap between segments and review flagged segments with the fraud and compliance teams. \
Small segments need more data before any conclusion is drawn.
""")
else:
    print("No categorical fields available for segment analysis.")

## 10. Decision storytelling and recommendations (Part 1, Q6 · Part 3)

### Figure 14 — Workload and cost: turning scores into an operating policy

**What it shows.** (Left) A cumulative gains curve: if analysts review the riskiest X% of transactions, what share of all fraud do they catch? (Right) Expected cost per 1,000 transactions at each threshold on validation data, for three assumed ratios between the cost of a missed fraud and the cost of a false alert. **Why it matters.** The data contains no real costs, so we show how the best threshold *moves* with the bank's own cost estimate. This turns the model into a decision tool rather than a single number.

In [ ]:
COST_RATIOS = [5, 10, 25]   # cost of one missed fraud ÷ cost of one false alert (illustrative assumptions)
order_idx = np.argsort(-test_scores)
cum_fraud = np.cumsum(yt[order_idx]) / yt.sum()
reviewed = np.arange(1, len(yt) + 1) / len(yt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(reviewed, cum_fraud, color="#C44E52", lw=2.5, label=final_name)
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random review")
axes[0].plot([0, y_test.mean(), 1], [0, 1, 1], color="grey", ls=":", label="Perfect model")
axes[0].scatter(test_op["Alert rate"], test_op["Recall"], color="black", s=80, zorder=5, label="Operating point")
gain_points = {}
for share in [0.10, 0.20]:
    k = int(share * len(yt)); gain_points[share] = cum_fraud[k - 1]
    axes[0].scatter(share, cum_fraud[k - 1], color="#C44E52", s=40, zorder=6)
    axes[0].text(share + 0.02, cum_fraud[k - 1] - 0.07, f"Top {share:.0%} → {cum_fraud[k - 1]:.0%} of fraud", fontsize=9)
axes[0].xaxis.set_major_formatter(PercentFormatter(1.0)); axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_xlabel("Share of transactions reviewed (riskiest first)"); axes[0].set_ylabel("Share of all fraud caught")
axes[0].set_title("Cumulative gains (test)"); axes[0].legend(fontsize=8, loc="lower right")

cost_rows = []
for r, col in zip(COST_RATIOS, ["#55A868", "#DD8452", "#C44E52"]):
    cost = (thr_df["FP"] + r * thr_df["FN"]) / len(y_val) * 1000
    t_star = float(thr_df.loc[cost.idxmin(), "threshold"])
    axes[1].plot(thr_df["threshold"], cost, color=col, lw=2, label=f"Missed fraud = {r}× false alert")
    axes[1].scatter(t_star, cost.min(), color=col, s=70, zorder=5)
    m = metrics_at(y_test, test_scores, t_star)
    cost_rows.append({"Cost ratio (FN:FP)": f"{r}:1", "Best threshold (validation)": t_star,
                      "Test recall": m["Recall"], "Test precision": m["Precision"],
                      "Alerts per 1,000": 1000 * m["Alert rate"], "Frauds missed per 1,000": 1000 * m["FN"] / len(yt)})
axes[1].set_xlabel("Decision threshold"); axes[1].set_ylabel("Expected cost per 1,000 transactions\n(in units of one false alert)")
axes[1].set_title("Cost-based threshold choice (validation)"); axes[1].legend(fontsize=8)
plt.tight_layout()
save_fig("fig14_gains_and_cost")
plt.show()

cost_table = pd.DataFrame(cost_rows)
display(cost_table.round(3))
md(f"""#### Interpretation
- Reviewing only the riskiest **10%** of transactions would catch **{gain_points[0.10]:.0%}** of all fraud, and the riskiest **20%** would catch **{gain_points[0.20]:.0%}** — \
compared with 10% and 20% for random review.
- As a missed fraud becomes more expensive relative to a false alert ({' → '.join(c['Cost ratio (FN:FP)'] for c in cost_rows)}), the best threshold falls \
({' → '.join(f"{c['Best threshold (validation)']:.2f}" for c in cost_rows)}) and alerts per 1,000 transactions rise \
({' → '.join(f"{c['Alerts per 1,000']:.0f}" for c in cost_rows)}).
- **Connection:** the bank should choose the threshold from *its own* loss and staffing figures. The model stays the same; only the policy changes.
""")

In [ ]:
alerts_per_1000 = 1000 * (test_op["TP"] + test_op["FP"]) / len(y_test)
caught_per_1000 = 1000 * test_op["TP"] / len(y_test)
missed_per_1000 = 1000 * test_op["FN"] / len(y_test)
top_drivers = list(shap_importance.index[:3]) if sv is not None else list(perm_df["feature"][:3])

md(f"""### Recommendations (supported by the findings above)

1. **Use the model as a risk-ranking triage tool, not an automatic block.** At threshold {OPERATING_THRESHOLD:.2f} it would send about \
**{alerts_per_1000:.0f} of every 1,000 transactions** to review, of which ~{caught_per_1000:.0f} are fraud, while ~{missed_per_1000:.0f} frauds per 1,000 still pass (Figures 4, 14). \
A human makes the final decision.
2. **Size the review team first, then set the threshold.** Figure 14 shows how alert volume and missed fraud move with the cost ratio. Start with the F1-optimal point and adjust \
once the bank supplies real costs for a missed fraud and for an analyst review.
3. **Tier the response by score.** Very high scores → hold and verify with the customer (step-up authentication); scores near the threshold → passive review; \
low scores → pass. This reduces friction for good customers (calibration and score separation, Figure 6).
4. **Show investigators the explanation with each alert.** The top drivers — {', '.join('`' + d + '`' for d in top_drivers)} — and the per-alert SHAP breakdown (Figure 11) \
let analysts confirm or dismiss an alert faster and give a reason when a customer asks.
5. {"**Prefer class weights over SMOTE** for this problem: equal or better ranking without synthetic transactions (Figure 2)." if v['SMOTE + LR'] <= v['Balanced LR'] + SELECTION_TOLERANCE else "**SMOTE helped here** (Figure 2), but synthetic fraud must be validated against real cases before use."}
6. {"**Invest in new information, not more tuning.** Missed frauds resemble legitimate activity on current features (Figure 12); customer history, device and merchant features are the most promising route to higher recall." if camouflaged else "**Test a non-linear model and richer features next.** Missed frauds still differ from legitimate activity (Figure 12), so more flexible models may recover them."}
""")

## 11. Real-world use, monitoring, risks and limitations (Part 3 · Part 1, Q7)

### How the model would be used
The model would score each transaction as it arrives. Transactions above the threshold join an **analyst review queue**, ranked by score, with the SHAP explanation attached. High-score cases could trigger a step-up check (one-time code, call-back) instead of a decline. Analyst decisions feed back as new labels for retraining.

### Figure 15 — Monitoring demonstration: Population Stability Index (PSI)

**What it shows.** PSI measures how much each feature's distribution (and the model score) has shifted between a reference period (training) and a new period (here, the test rows). Common rules of thumb: PSI < 0.10 stable, 0.10–0.25 watch, > 0.25 investigate. **Why it matters.** Fraud patterns change. Fraud labels arrive weeks later (chargebacks, investigations), so **input drift is the earliest warning** that the model may be going stale. Here the "new period" comes from the same random split, so we expect low PSI; this demonstrates the tool the bank would run every month.

In [ ]:
def psi(expected, actual, bins=10):
    expected, actual = np.asarray(expected, float), np.asarray(actual, float)
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    e = np.histogram(np.clip(expected, edges[0], edges[-1]), edges)[0] / len(expected)
    a = np.histogram(np.clip(actual, edges[0], edges[-1]), edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

psi_vals = {c: psi(X_train[c].dropna(), X_test[c].dropna()) for c in numeric_cols}
psi_vals["MODEL SCORE"] = psi(train_scores, test_scores)
psi_s = pd.Series(psi_vals).sort_values(ascending=True)

plt.figure(figsize=(8, 0.32 * len(psi_s) + 1.5))
plt.barh(psi_s.index, psi_s.values,
         color=["#C44E52" if v > 0.25 else "#DD8452" if v > 0.10 else "#55A868" for v in psi_s.values])
plt.axvline(0.10, color="#DD8452", ls="--", lw=1, label="0.10 watch")
plt.axvline(0.25, color="#C44E52", ls="--", lw=1, label="0.25 investigate")
plt.xlabel("PSI (training → new data)"); plt.title("Drift monitoring demo: training vs test period")
plt.legend(fontsize=8); plt.tight_layout()
save_fig("fig15_psi_monitoring")
plt.show()

md(f"""#### Interpretation
- Largest feature shift: `{psi_s.drop('MODEL SCORE').idxmax()}` (PSI {psi_s.drop('MODEL SCORE').max():.3f}); model-score PSI {psi_vals['MODEL SCORE']:.3f}. \
{"All values are below 0.10, as expected for a random split." if psi_s.max() < 0.10 else "Some values exceed 0.10 and would be reviewed in production."}
- **Connection:** in production this chart would compare each month with the training period and trigger a review before performance visibly drops.
""")

### Monitoring plan

| What to monitor | How often | Trigger for action |
|---|---|---|
| Feature and score drift (PSI, Figure 15) | Weekly / monthly | PSI > 0.25 on a top driver or on the score |
| Alert volume and alert rate | Daily | Outside the range the review team can handle |
| Precision of reviewed alerts (analyst outcomes) | Weekly | Falls well below the validation level |
| Recall from confirmed fraud and chargebacks (labels arrive late) | Monthly | Falls more than ~10 points below test recall |
| Segment recall and false-positive rates (Figure 13) | Monthly | Gap between segments beyond an agreed limit |
| Calibration (Figure 6) | Quarterly | Scores drift away from observed fraud rates |

Retrain on recent labelled data on a fixed schedule (e.g. quarterly) or when a trigger fires, re-run this notebook's validation and test steps, and keep the previous model as a fallback.

### Risks

- **Overfitting.** Controlled with L1/L2 regularization, validation-only model and threshold selection, and repeated cross-validation; the train–test gap in §7 is our evidence. Risk remains that the threshold is tuned to one validation set, so it should be re-checked on fresh data.
- **Data leakage.** All preprocessing and SMOTE run inside pipelines fitted on training rows; the test set was opened once, after every decision. The single-feature AUC check found no obviously leaking column, but the bank must confirm that every feature is available **at the moment of the transaction** (no fields filled in after an investigation).
- **Bias and unequal treatment.** The model can inherit bias in historical fraud labels (e.g. some channels investigated more often). Segment analysis (Figure 13) is the first check; demographic attributes are not in this file, so a fuller fairness audit is not possible here.
- **Unreliable predictions.** Scores are not calibrated probabilities (Figure 6); camouflaged fraud is systematically missed (Figure 12); and new fraud tactics will not resemble training fraud. Human review and drift monitoring are the safeguards.
- **Synthetic data (if SMOTE were chosen).** Synthetic fraud can create unrealistic "bridges" between real fraud and legitimate transactions and can distort probabilities, which is one reason class weights were preferred.

### Ethical considerations
- **Human in the loop:** no customer should be blocked or reported solely because of this score.
- **Transparency and contestability:** customers whose transactions are held should receive a reason and a quick way to resolve it; SHAP explanations make this practical.
- **Privacy:** transaction data is personal financial data; use, retention and access must follow applicable privacy law (for example PIPEDA in Canada or GDPR in the EU) and the bank's data-governance rules.
- **Proportionality:** customers in segments with higher false-positive rates bear more friction; this burden should be measured and justified.

### Limitations of the data and the model
- One static sample of 10,000 transactions from a public Kaggle dataset — we do not know the bank, time period or how fraud was labelled, and the data may be synthetic or simplified.
- No timestamps were used for an out-of-time test, so the test set measures performance on *similar* transactions, not on *future* ones.
- Logistic regression captures only linear effects in log-odds; interactions and thresholds in real fraud are only partly captured (the random-forest benchmark in §5 quantifies how much may be missing).
- Costs, review capacity and customer impact are assumptions (§10), not data.
- Fraud labels may be incomplete: undetected fraud labelled as legitimate would make every metric here optimistic.

### Future improvements
1. **Out-of-time validation**: train on earlier months, test on later months.
2. **Richer behavioural features**: customer history, transaction velocity, device and merchant risk.
3. **Non-linear models** (gradient boosting) with SHAP explanations, compared against this baseline under the same selection rule.
4. **Probability calibration** and **cost-sensitive thresholds** using the bank's real costs.
5. **Fairness audit** across customer groups with appropriate, legally permitted attributes.
6. **Feedback loop** from analyst decisions for regular retraining, plus champion–challenger testing before replacing a live model.

## 12. Auto-generated summary for the final report and presentation

The cell below writes the key numbers into a short narrative that can be pasted into the 5–8 page report, plus a 4-slide outline for the 3-minute presentation. It then exports all figures and tables.

In [ ]:
summary = {
    "final_model": final_name, "settings": final["note"], "operating_threshold": OPERATING_THRESHOLD,
    "fraud_share": float(y_all.mean()), "n_transactions": int(len(df)),
    "test": {k: (float(v) if isinstance(v, (float, np.floating)) else int(v)) for k, v in test_op.items()},
    "validation_pr_auc_by_model": val_table.set_index("Model")["PR-AUC"].round(4).to_dict(),
    "cv_pr_auc_mean_std": [float(fm), float(fs)],
    "top_drivers": top_drivers,
}
with open(os.path.join(OUT_DIR, "ms5_summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2)

story = f"""### Report narrative (draft)

**Problem.** Banks must find fraud among many legitimate transactions without overwhelming analysts or inconveniencing customers. \
We built a model that ranks transactions by fraud risk to support analyst review.

**Data.** {len(df):,} transactions, {pct(y_all.mean())} fraud, no missing values or duplicates. Accuracy is misleading at this imbalance, \
so we used recall, precision, F1 and PR-AUC. The strongest single signals are {', '.join('`'+u+'`' for u in uni_df['feature'][:3])}; \
{"no leaking column was found." if not suspects else f"possible leakage in {suspects} must be checked."}

**Model selection.** We compared {len(candidates)} candidates on validation PR-AUC. **{final_name}** ({final['note']}) was chosen as the simplest model within 0.01 of the best \
(validation PR-AUC {v[final_name]:.3f}; baseline {v['Baseline LR']:.3f}; SMOTE {v['SMOTE + LR']:.3f}; random forest {v['Random forest (benchmark)']:.3f}). \
{"Class weights matched or beat SMOTE without synthetic data." if v['SMOTE + LR'] <= v['Balanced LR'] + SELECTION_TOLERANCE else "SMOTE improved ranking on validation."}

**Performance.** On the untouched test set at threshold {OPERATING_THRESHOLD:.2f}: recall {pct(test_op['Recall'])}, precision {pct(test_op['Precision'])}, \
F1 {test_op['F1']:.3f}, PR-AUC {test_op['PR-AUC']:.3f} (chance {y_test.mean():.3f}), ROC-AUC {test_op['ROC-AUC']:.3f}. \
Repeated CV gives PR-AUC {fm:.3f} ± {fs:.3f}, and the train–test gap is {gap:+.3f}{", so results are stable and not overfitted." if abs(gap) < 0.05 and fs < 0.03 else "; some instability or overfitting remains."}

**Insights.** SHAP, odds ratios and permutation importance point to {', '.join('`'+d+'`' for d in top_drivers)} as the main drivers of the predictions. \
{"Missed frauds resemble legitimate transactions on these features, so further gains need new information rather than more tuning." if camouflaged else "Missed frauds still differ from legitimate ones, so non-linear models and richer features are the next step."}

**Recommendations.** Deploy as a triage tool with human review: about {alerts_per_1000:.0f} alerts per 1,000 transactions, with the threshold set from the bank's \
own cost of missed fraud vs false alerts. Attach SHAP explanations to alerts; monitor drift, alert precision and segment fairness monthly.

**Limitations.** One static public sample, no out-of-time test, linear effects only, assumed costs, and possibly incomplete fraud labels.
"""
slides = f"""### Suggested 3-minute slide outline (4 slides)
1. **Problem & data** — why fraud triage matters; {pct(y_all.mean())} fraud; accuracy is misleading. *(Figure 1, panel A)*
2. **Model & performance** — {final_name}; test recall {pct(test_op['Recall'])}, precision {pct(test_op['Precision'])}, PR-AUC {test_op['PR-AUC']:.2f}. *(Figure 4 or Figure 5)*
3. **What drives fraud alerts** — top drivers and one explained alert. *(Figure 8, optionally Figure 11)*
4. **Recommendation** — triage tool with human review; reviewing the top 10% catches {gain_points[0.10]:.0%} of fraud; threshold from real costs; monitor drift and fairness. *(Figure 14)*
"""
md(story)
md(slides)
with open(os.path.join(OUT_DIR, "report_narrative_and_slide_outline.md"), "w") as fh:
    fh.write(story + "\n\n" + slides)

In [ ]:
val_table.to_csv(os.path.join(OUT_DIR, "validation_model_comparison.csv"), index=False)
final_scores.to_csv(os.path.join(OUT_DIR, "final_train_val_test_scores.csv"), index=False)
cost_table.to_csv(os.path.join(OUT_DIR, "cost_threshold_scenarios.csv"), index=False)
perm_df.to_csv(os.path.join(OUT_DIR, "permutation_importance.csv"), index=False)
if sv is not None:
    shap_importance.rename("mean_abs_shap").to_csv(os.path.join(OUT_DIR, "shap_importance.csv"))
if seg_tables:
    seg_all.to_csv(os.path.join(OUT_DIR, "segment_performance.csv"), index=False)

os.makedirs("ms5_results", exist_ok=True)
for folder in [FIG_DIR, OUT_DIR]:
    shutil.copytree(folder, os.path.join("ms5_results", folder), dirs_exist_ok=True)
archive = shutil.make_archive("ms5_results", "zip", "ms5_results")
print(f"Saved {len(FIGURES)} figures and {len(os.listdir(OUT_DIR))} output files to {archive}")

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Not running in Colab — download", archive, "manually.")

## 13. Conclusion

We set out to help a bank find fraud without flooding its analysts or blocking good customers. Starting from Milestone 4, we corrected the evaluation pipeline, tested six candidate models under a selection rule fixed in advance, and chose the simplest model whose ranking quality was within reach of the best. The decision threshold was chosen on validation data, and the untouched test set confirmed that performance carries over to unseen transactions; repeated cross-validation showed it is stable.

Three explainability methods agree on which signals drive the alerts, and the error analysis shows the model's main weakness: fraud that looks like ordinary activity on the available features. The model is therefore best used as a **transparent triage tool with human review**, with a threshold set from the bank's own costs and with ongoing monitoring of drift, alert quality and fairness. Before any real deployment it needs out-of-time testing on recent data, richer behavioural features, and a full fairness and privacy review.